# Diagnostico de Neumonia con Transfer Learning

Notebook autocontenido para entrenar un modelo basado en `ResNet-50` pre-entrenada y adaptada a dos clases: `NORMAL` y `PNEUMONIA`.

> Uso educativo. No reemplaza evaluacion medica ni validacion clinica.

## 1. Importaciones y configuracion

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import random

import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 42
DATA_DIR = Path("data")
OUTPUT_DIR = Path("models")
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
NUM_WORKERS = 0
USE_PRETRAINED = True
FREEZE_BACKBONE = True

CLASS_TO_INDEX = {"NORMAL": 0, "PNEUMONIA": 1}
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dispositivo: {DEVICE}")

In [ ]:
# CELDA OPCIONAL PARA KAGGLE
# Ejecuta esta celda solo si estas corriendo el notebook en Kaggle.
# En Kaggle los datasets se montan en /kaggle/input y no en ./data.

KAGGLE_INPUT_ROOT = Path("/kaggle/input")

if not KAGGLE_INPUT_ROOT.exists():
    raise RuntimeError("No se detecto /kaggle/input. Omite esta celda si estas trabajando localmente.")

def has_expected_dataset_structure(path: Path) -> bool:
    required_paths = [
        path / "train" / "NORMAL",
        path / "train" / "PNEUMONIA",
        path / "val" / "NORMAL",
        path / "val" / "PNEUMONIA",
    ]
    return all(required_path.exists() for required_path in required_paths)

candidate_data_dirs = [
    path
    for path in KAGGLE_INPUT_ROOT.rglob("*")
    if path.is_dir() and has_expected_dataset_structure(path)
]

if not candidate_data_dirs:
    raise FileNotFoundError(
        "No se encontro un dataset con train/val y clases NORMAL/PNEUMONIA dentro de /kaggle/input. "
        "Revisa que el dataset este agregado como input del notebook."
    )

preferred_data_dirs = [path for path in candidate_data_dirs if path.name.lower() == "chest_xray"]
DATA_DIR = sorted(preferred_data_dirs or candidate_data_dirs, key=lambda path: str(path))[0]

print(f"DATA_DIR configurado para Kaggle: {DATA_DIR}")
print("Splits encontrados:", sorted(path.name for path in DATA_DIR.iterdir() if path.is_dir()))


## 2. Dataset, transformaciones y DataLoaders

Si `USE_PRETRAINED = True`, la primera ejecucion puede descargar los pesos de ResNet-50.

In [ ]:
@dataclass(frozen=True)
class DataLoaderConfig:
    data_dir: Path
    batch_size: int = 16
    num_workers: int = 0
    image_size: int = 224
    pin_memory: bool = False


class PneumoniaDataset(Dataset):
    def __init__(self, root_dir: str | Path, transform: transforms.Compose | None = None) -> None:
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = self._collect_samples()
        if not self.samples:
            raise ValueError(f"No se encontraron imagenes validas en: {self.root_dir}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

    def _collect_samples(self) -> list[tuple[Path, int]]:
        samples: list[tuple[Path, int]] = []
        for class_name, class_index in CLASS_TO_INDEX.items():
            class_dir = self.root_dir / class_name
            if not class_dir.exists():
                continue
            for image_path in class_dir.rglob("*"):
                if image_path.suffix.lower() in VALID_EXTENSIONS:
                    samples.append((image_path, class_index))
        return sorted(samples, key=lambda sample: str(sample[0]))


def build_transforms(image_size: int = 224, train: bool = True) -> transforms.Compose:
    pipeline = [transforms.Resize((image_size, image_size))]
    if train:
        pipeline.extend([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
        ])
    pipeline.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return transforms.Compose(pipeline)


def create_dataloaders(config: DataLoaderConfig) -> dict[str, DataLoader]:
    settings = {
        "train": {"shuffle": True, "train": True},
        "val": {"shuffle": False, "train": False},
        "test": {"shuffle": False, "train": False},
    }
    dataloaders: dict[str, DataLoader] = {}
    for split_name, split_settings in settings.items():
        split_dir = config.data_dir / split_name
        if split_dir.exists():
            dataset = PneumoniaDataset(split_dir, build_transforms(config.image_size, split_settings["train"]))
            dataloaders[split_name] = DataLoader(
                dataset,
                batch_size=config.batch_size,
                shuffle=split_settings["shuffle"],
                num_workers=config.num_workers,
                pin_memory=config.pin_memory,
            )
    missing_required = {"train", "val"} - set(dataloaders)
    if missing_required:
        missing = ", ".join(sorted(missing_required))
        raise FileNotFoundError(f"Faltan splits requeridos en {config.data_dir}: {missing}")
    return dataloaders

## 3. Modelo con ResNet-50

Se reemplaza la capa final de ResNet-50 por una salida de dos clases. Con `FREEZE_BACKBONE = True` solo se entrena la cabeza clasificadora.

In [ ]:
def build_transfer_model(num_classes: int = 2, pretrained: bool = True, freeze_backbone: bool = True) -> nn.Module:
    weights = models.ResNet50_Weights.DEFAULT if pretrained else None
    model = models.resnet50(weights=weights)

    if freeze_backbone:
        for parameter in model.parameters():
            parameter.requires_grad = False

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes),
    )
    return model


model = build_transfer_model(
    num_classes=len(CLASS_TO_INDEX),
    pretrained=USE_PRETRAINED,
    freeze_backbone=FREEZE_BACKBONE,
).to(DEVICE)

trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
print(f"Parametros entrenables: {trainable_parameters:,} / {total_parameters:,}")
model

## 4. Entrenamiento y evaluacion

In [ ]:
def train_one_epoch(model: nn.Module, dataloader: DataLoader, criterion: nn.Module, optimizer: optim.Optimizer, device: torch.device) -> tuple[float, float]:
    model.train()
    running_loss = 0.0
    all_predictions: list[int] = []
    all_targets: list[int] = []
    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        all_predictions.extend(torch.argmax(outputs, dim=1).detach().cpu().tolist())
        all_targets.extend(labels.detach().cpu().tolist())
    return running_loss / len(dataloader.dataset), accuracy_score(all_targets, all_predictions)


def evaluate(model: nn.Module, dataloader: DataLoader, criterion: nn.Module, device: torch.device) -> tuple[float, float, list[int], list[int]]:
    model.eval()
    running_loss = 0.0
    all_predictions: list[int] = []
    all_targets: list[int] = []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            all_predictions.extend(torch.argmax(outputs, dim=1).cpu().tolist())
            all_targets.extend(labels.cpu().tolist())
    return running_loss / len(dataloader.dataset), accuracy_score(all_targets, all_predictions), all_predictions, all_targets


def plot_history(history: dict[str, list[float]], output_path: Path) -> None:
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train")
    plt.plot(epochs, history["val_loss"], label="Validacion")
    plt.xlabel("Epoca")
    plt.ylabel("Perdida")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train")
    plt.plot(epochs, history["val_acc"], label="Validacion")
    plt.xlabel("Epoca")
    plt.ylabel("Exactitud")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path)
    plt.show()

## 5. Carga de datos y entrenamiento

In [ ]:
dataloaders = create_dataloaders(
    DataLoaderConfig(
        data_dir=DATA_DIR,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        image_size=IMAGE_SIZE,
        pin_memory=DEVICE.type == "cuda",
    )
)

for split_name, dataloader in dataloaders.items():
    print(f"{split_name}: {len(dataloader.dataset)} imagenes")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam((parameter for parameter in model.parameters() if parameter.requires_grad), lr=LEARNING_RATE)
best_model_path = OUTPUT_DIR / "best_resnet50_transfer.pth"
best_val_accuracy = -1.0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, dataloaders["train"], criterion, optimizer, DEVICE)
    val_loss, val_acc, _, _ = evaluate(model, dataloaders["val"], criterion, DEVICE)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    print(f"Epoca {epoch:03d}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save({"model_name": "resnet50_transfer", "model_state_dict": model.state_dict(), "best_val_accuracy": best_val_accuracy, "image_size": IMAGE_SIZE, "class_to_index": CLASS_TO_INDEX, "use_pretrained": USE_PRETRAINED, "freeze_backbone": FREEZE_BACKBONE}, best_model_path)
        print(f"Nuevo mejor modelo guardado en: {best_model_path}")

print(f"Mejor exactitud de validacion: {best_val_accuracy:.4f}")
plot_history(history, OUTPUT_DIR / "training_curves_resnet50_transfer.png")

## 6. Evaluacion final en test

In [ ]:
if best_model_path.exists():
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])

if "test" in dataloaders:
    test_loss, test_acc, predictions, targets = evaluate(model, dataloaders["test"], criterion, DEVICE)
    print(f"Test loss={test_loss:.4f} | Test acc={test_acc:.4f}")
    print(classification_report(targets, predictions, target_names=["NORMAL", "PNEUMONIA"], digits=4))
else:
    print("No se encontro split test. Evaluacion final omitida.")